# ASML options in Amsterdam
Run all cells with the project's `.venv` kernel. Each refresh reads Amsterdam ASML share and option executions from Euronext's delayed trade files. The tables show Amsterdam local time and euros per share. Regular continuous trading in the ASL class runs from 09:01 to 17:30 Amsterdam time. Allow about 15 minutes after the open for the first published trades.

[Market settings](../config/asml_amsterdam.toml) · [Contract specification](https://live.euronext.com/en/product/stock-options/ASL-DAMS/contract-specification) · [Euronext delayed files](https://marketdata.euronext.com/data-reporting-service/trades-file)

In [ ]:
REFRESH = True  # False reads the latest local capture offline.
ROWS = 8

In [ ]:
from pathlib import Path
import sqlite3
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / "config/asml_amsterdam.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the trading project folder in VS Code and run the notebook again.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from options_lab.cli import parser, run
from options_lab.config import load_settings
from modules.storage_reporting.amsterdam import amsterdam_database, read_report, recent_windows, write_trade_report

PROFILE = ROOT / "config/asml_amsterdam.toml"
DATA = ROOT / "data"
REPORTS = ROOT / "reports"
settings = load_settings(PROFILE)
if not isinstance(ROWS, int) or ROWS < 1:
    raise ValueError("ROWS must be a positive integer.")


def local_time(series):
    return pd.to_datetime(series, utc=True).dt.tz_convert("Europe/Amsterdam").dt.strftime("%Y-%m-%d %H:%M:%S")

## Collection status

In [ ]:
if REFRESH:
    args = parser().parse_args(["--config", str(PROFILE), "--data-dir", str(DATA), "collect"])
    try:
        run(args)
    except (OSError, ValueError, KeyError, sqlite3.Error) as error:
        print(f"Refresh failed ({error}). Showing the latest saved capture if available.")
else:
    print("Offline mode. Showing stored Amsterdam trades.")

with amsterdam_database(DATA) as db:
    capture, records = read_report(db, settings)
    captures = pd.DataFrame(recent_windows(db, settings, ROWS))

if captures.empty:
    print("No Amsterdam captures are stored. Set REFRESH = True to collect a trade window.")
else:
    for col in ["retrieved_utc", "newest_trade_utc"]:
        captures[col.replace("_utc", "_amsterdam")] = local_time(captures.pop(col))
    display(captures)

## Most recent ASL executions
Each row is a reported exchange trade. Quantity is in contracts. The share price comes from the latest ASML share trade already published when the option traded.

In [ ]:
trades = pd.DataFrame(records)
if not trades.empty:
    trades["traded_amsterdam"] = local_time(trades["traded_utc"])
    trades["expiry_date"] = pd.to_datetime(trades["expiry_utc"], utc=True).dt.tz_convert("Europe/Amsterdam").dt.date
    display(trades[["traded_amsterdam", "option_isin", "kind", "expiry_date", "strike_eur",
                    "quantity", "trade_eur", "stock_eur"]].head(ROWS))
elif capture:
    print(f"No eligible ASL executions in the saved {capture['window']} window on {capture['trading_day']}.")

## Model values at execution time
The European Black–Scholes–Merton model approximates these American-style options. It uses a fixed volatility, rate and dividend yield. The tape provides executed prices. Bid/ask spreads and simulated fills require quote data.

In [ ]:
model = settings.model
print(f"EUR scenarios · volatility {model.volatility:.1%} · rate {model.rate:.1%} · dividend yield {model.dividend_yield:.1%}")
if not trades.empty:
    display(trades[["traded_amsterdam", "option_isin", "trade_eur", "model_eur",
                    "stock_age_s", "valuation_status"]].head(ROWS).round(3))
if capture:
    report = write_trade_report(REPORTS, capture, records, settings)
    print(f"Saved report: {report.relative_to(ROOT) if report.is_relative_to(ROOT) else report}")